In [15]:
# Centralized inputs for this notebook.
# Edit prepare_mgp_config.py, or set MGP_DATA_ROOT,
# MGP_HUMANN_DATASET, and MGP_EDGE_DATASET before starting Jupyter.

import importlib.util
import sys
from pathlib import Path

_CWD = Path.cwd().resolve()
_CONFIG_CANDIDATES = [
    _CWD / "prepare_mgp_config.py",
    _CWD / "data_process" / "prepare_mgp_config.py",
    _CWD / "MGPAN" / "data_process" / "prepare_mgp_config.py",
]

for _config_path in _CONFIG_CANDIDATES:
    if _config_path.exists():
        _config_path = _config_path.resolve()
        break
else:
    searched = "\n".join(str(path) for path in _CONFIG_CANDIDATES)
    raise FileNotFoundError(
        f"Cannot locate prepare_mgp_config.py. Searched:\n{searched}"
    )

# Load by absolute file path so a stale module in the kernel cannot shadow it.
sys.modules.pop("prepare_mgp_config", None)
_spec = importlib.util.spec_from_file_location("prepare_mgp_config", _config_path)
if _spec is None or _spec.loader is None:
    raise ImportError(f"Cannot load prepare_mgp_config.py from {_config_path}")
cfg = importlib.util.module_from_spec(_spec)
sys.modules["prepare_mgp_config"] = cfg
_spec.loader.exec_module(cfg)

print(f"Loaded config: {_config_path}")
cfg.print_config()


Loaded config: D:\coursework\研究生\code\MGPAN\data_process\prepare_mgp_config.py
PROJECT_ROOT: D:\coursework\研究生\code
PACKAGE_ROOT: D:\coursework\研究生\code\MGPAN
DATA_ROOT: D:\coursework\研究生\code\MGPAN\data
HUMAnN dataset: QinN_2014 -> D:\coursework\研究生\code\MGPAN\data\QinN_2014
Edge dataset: QinN_2014 -> D:\coursework\研究生\code\MGPAN\data\QinN_2014
P-P input: D:\coursework\研究生\code\MGPAN\data\QinN_2014\pathway_abundance_abundance.csv
M-M input: D:\coursework\研究生\code\MGPAN\data\QinN_2014\relative_abundance_abundance.csv
P-P output dir: D:\coursework\研究生\code\MGPAN\data\QinN_2014\pathway
M-M output dir: D:\coursework\研究生\code\MGPAN\data\QinN_2014\microbe
Metadata: D:\coursework\研究生\code\MGPAN\data\QinN_2014\relative_abundance_metadata.csv
Subject IDs: D:\coursework\研究生\code\MGPAN\data\QinN_2014\subject_ids.csv
Raw graph: D:\coursework\研究生\code\MGPAN\data\QinN_2014\QN_graphdata.bin
Raw graph metadata: D:\coursework\研究生\code\MGPAN\data\QinN_2014\QN_graphdata_meta.pkl
Pruned graph: D:\coursew

## 1. Convert CSV to TSV


In [3]:
import pandas as pd

pathway_df = pd.read_csv(cfg.PATHWAY_ABUNDANCE_CSV)
pathway_df.to_csv(cfg.PATHWAY_ABUNDANCE_TSV, sep="	", index=False)
print(f"Saved pathway abundance TSV: {cfg.PATHWAY_ABUNDANCE_TSV}")

gene_df = pd.read_csv(cfg.GENE_FAMILIES_CSV)
gene_df.to_csv(cfg.GENE_FAMILIES_TSV, sep="	", index=False)
print(f"Saved gene family TSV: {cfg.GENE_FAMILIES_TSV}")


Saved pathway abundance TSV: D:\coursework\研究生\code\MGPAN\data\NielsenHB_2014\pathway_abundance_abundance.tsv
Saved gene family TSV: D:\coursework\研究生\code\MGPAN\data\NielsenHB_2014\gene_families_abundance.tsv


## 2. Extract microbe-pathway-UniRef90 relations


In [16]:
#参考readme.md中说明来生成pathway-taxonomy-gene mapping（path_taxonomy_uf90.tsv）
# 生成pathway-taxonomy-gene mapping（path_taxonomy_uf90.tsv）
# 该文件由 humann_merge_abundance 脚本生成#

import csv

input_file = cfg.PATH_TAXONOMY_TSV
output_file = cfg.PATH_TAXONOMY_CSV

with open(input_file, "r", encoding="utf-8") as fin, open(
    output_file,
    "w",
    newline="",
    encoding="utf-8",
) as fout:
    writer = csv.writer(fout)
    writer.writerow(["pathway", "Taxonomy", "gene"])

    for line in fin:
        line = line.strip()
        if not line:
            continue
        first_col = line.split("	")[0]
        parts = first_col.split("|")
        pathway = parts[0] if len(parts) > 0 else ""
        taxonomy = parts[1] if len(parts) > 1 else ""
        gene = parts[2] if len(parts) > 2 else ""
        writer.writerow([pathway, taxonomy, gene])

print(f"Saved pathway-taxonomy-gene mapping: {output_file}")


Saved pathway-taxonomy-gene mapping: D:\coursework\研究生\code\MGPAN\data\QinN_2014\path-tax-uf90.csv


## 3. Save sample-level triplets


In [17]:
import pandas as pd
import os

# -------------------------------
# 1. 设置路径并读入文件
# -------------------------------
genes = pd.read_csv(cfg.GENE_FAMILIES_CSV, index_col=0)
pathways = pd.read_csv(cfg.PATHWAY_ABUNDANCE_CSV, index_col=0)
taxa = pd.read_csv(cfg.MICROBE_ABUNDANCE_CSV, index_col=0)
mapping = pd.read_csv(cfg.PATH_TAXONOMY_CSV)

# -------------------------------
# 2. 清理 mapping
# -------------------------------
mapping = mapping.dropna(subset=["gene", "Taxonomy", "pathway"])
mapping["gene"] = mapping["gene"].astype(str).str.strip().str.replace('"','').str.replace("'",'')
mapping["pathway"] = mapping["pathway"].astype(str).str.strip()
mapping["Taxonomy"] = mapping["Taxonomy"].astype(str).str.strip()

# -------------------------------
# 3. 转换 gene family abundance -> 长表
# -------------------------------
genes_long = genes.reset_index().melt(id_vars="index", var_name="Sample", value_name="GeneAbundance")
genes_long = genes_long.rename(columns={"index": "gene"})
genes_long["gene"] = genes_long["gene"].astype(str).str.strip()

# -------------------------------
# 4. 定义函数统一提取 g__属.s__种
# -------------------------------
def extract_gs(tax):
    parts = str(tax).split("|")
    gs = []
    for part in parts:
        if part.startswith("g__") or part.startswith("s__"):
            gs.append(part)
    return ".".join(gs)

# -------------------------------
# 5. 处理 taxa -> 长表
# -------------------------------
taxa.index = taxa.index.map(extract_gs)
taxa_long = taxa.reset_index().melt(id_vars="index", var_name="Sample", value_name="TaxonomyAbundance")
taxa_long = taxa_long.rename(columns={"index": "Taxonomy"})
taxa_long["Taxonomy"] = taxa_long["Taxonomy"].astype(str).str.strip()

# -------------------------------
# 6. 处理 pathways -> 长表，并拆分 pathway 和 Taxonomy
# -------------------------------
pathways_long = pathways.reset_index().melt(
    id_vars="index", var_name="Sample", value_name="PathwayAbundance"
)
pathways_long = pathways_long.rename(columns={"index": "pathway_full"})

# 拆分 pathway 与 Taxonomy
split_df = pathways_long["pathway_full"].str.split("|", n=1, expand=True)
pathways_long["pathway"] = split_df[0].str.split(":", n=1).str[0].str.strip()
pathways_long["Taxonomy"] = split_df[1].astype(str).map(extract_gs).str.strip()
pathways_long = pathways_long[["Sample","pathway","Taxonomy","PathwayAbundance"]].drop_duplicates()

# -------------------------------
# 7. 分批 merge 按 Sample 处理，内存友好
# -------------------------------
samples = genes_long["Sample"].unique()
result_chunks = []
ABUND_TH = cfg.TRIPLET_ABUNDANCE_THRESHOLD
for s in samples:
    genes_s = genes_long[genes_long["Sample"] == s]
    map_s = mapping[mapping["gene"].isin(genes_s["gene"])]
    
    merged_s = genes_s.merge(map_s, on="gene", how="inner")
    
    path_s = pathways_long[pathways_long["Sample"] == s]
    merged_s = merged_s.merge(path_s, on=["pathway","Taxonomy","Sample"], how="left")
    
    taxa_s = taxa_long[taxa_long["Sample"] == s]
    merged_s = merged_s.merge(taxa_s, on=["Taxonomy","Sample"], how="left")
    
    # 只保留 GeneAbundance>0 且 PathwayAbundance>0
    merged_s = merged_s[
        (merged_s["GeneAbundance"] > ABUND_TH) &
        (merged_s["PathwayAbundance"] > 0) &
        (merged_s["TaxonomyAbundance"] > 0)
    ]
    
    result_chunks.append(merged_s)

# 拼接所有样本
final = pd.concat(result_chunks, ignore_index=True)

# -------------------------------
# 8. 输出结果
# -------------------------------
final = final[["Sample", "pathway", "Taxonomy", "gene", 
               "GeneAbundance", "PathwayAbundance", "TaxonomyAbundance"]]
final = final.dropna(subset=["TaxonomyAbundance"])
final.to_csv(cfg.SAMPLE_PATH_TAX_GENE_CSV, index=False)
print("Finished! Result saved to sample_path_tax_gene_with_taxa1.csv")


Finished! Result saved to sample_path_tax_gene_with_taxa1.csv


## 4. Process triplets and edge weights


In [18]:
import os
import pandas as pd

# 基础路径
base_path = cfg.HUMANN_DIR

# 输入文件路径
input_file = os.path.join(base_path, "sample_path_tax_gene.csv")

# 输出文件路径
output_file = os.path.join(base_path, "sample_path_tax_gene_weight.csv")

# 读取数据
df = pd.read_csv(input_file)
df= df.dropna(subset=["TaxonomyAbundance"])
# 计算每个基因在所有菌种中的总丰度
# gene_total_abundance = df.groupby('gene')['GeneAbundance'].sum()

# # 计算每条通路包含的基因数
# pathway_gene_count = df.groupby('pathway')['gene'].nunique()
EPS = cfg.WEIGHT_EPS
# 计算 Microbe -> Gene 边权
# df['weight_microbe_gene'] = df.apply(
#     lambda row: row['GeneAbundance'] / gene_total_abundance[row['gene']]
#     if gene_total_abundance[row['gene']] > 0 else 0, axis=1
# )

df['weight_mg'] = df.groupby(['Sample','Taxonomy'])['GeneAbundance'] \
                     .transform(lambda x: x / (x.sum() + EPS))

# --- 2. Gene -> Microbe ---
# 对每个样本、每个基因内部按微生物归一化
df['weight_gm'] = df.groupby(['Sample','gene'])['TaxonomyAbundance'] \
                     .transform(lambda x: x / (x.sum() + EPS))

# --- 3. Gene -> Pathway ---
# 对每个样本、每个基因内部按通路归一化
df['weight_gp'] = df.groupby(['Sample','gene'])['PathwayAbundance'] \
                     .transform(lambda x: x / (x.sum() + EPS))

# --- 4. Pathway -> Gene ---
# 对每个样本、每条通路内部按基因归一化
df['weight_pg'] = df.groupby(['Sample','pathway'])['GeneAbundance'] \
                     .transform(lambda x: x / (x.sum() + EPS))


# 计算 Gene -> Pathway 边权
# df['weight_gene_pathway'] = df['pathway'].apply(
#     lambda p: 1 / pathway_gene_count[p] if pathway_gene_count[p] > 0 else 0
# )

# 保存为新的 CSV 文件
df.to_csv(output_file, index=False)
print("完成！文件已保存到：", output_file)

print("完成！新文件已保存为：sample_path_tax_gene_with_taxa_with_weight1.csv")


完成！文件已保存到： D:\coursework\研究生\code\MGPAN\data\QinN_2014\sample_path_tax_gene_weight.csv
完成！新文件已保存为：sample_path_tax_gene_with_taxa_with_weight1.csv


## 5. Clean microbe and pathway names


In [7]:
import os
import pandas as pd
import re

# ==========================================================
# 1. 微生物名清洗函数
# ==========================================================
def convert_abund_rowname_microbe(rowname):
    """
    将 rowname 转为 g__Genus.s__Species 格式
    """
    if isinstance(rowname, str) and rowname.count('g__') == 1 and rowname.count('s__') == 1 and '|' not in rowname:
        return rowname.strip()

    match_g = re.search(r'g__[^|.]+', rowname)
    match_s = re.search(r's__[^|.]+', rowname)
    if match_g and match_s:
        return f"{match_g.group(0)}.{match_s.group(0)}"
    elif match_g:
        return match_g.group(0)
    else:
        return None


# ==========================================================
# 2. 通路名清洗函数
# ==========================================================
def convert_abund_rowname_pathway(rowname):
    """
    通路名只保留冒号前面的部分（如果有冒号）
    """
    if isinstance(rowname, str):
        return rowname.split(":", 1)[0].strip()
    return None


# ==========================================================
# 3. 文件路径
# ==========================================================
microbe_abund_file = cfg.MICROBE_ABUNDANCE_CSV
pathway_abund_file = cfg.PATHWAY_ABUNDANCE_CSV

microbe_abund_new_file = cfg.MICROBE_ABUNDANCE_CLEANED_CSV
pathway_abund_new_file = cfg.PATHWAY_ABUNDANCE_CLEANED_CSV


# ==========================================================
# 4. 清洗微生物丰度表
# ==========================================================
df_micro = pd.read_csv(microbe_abund_file, index_col=0)

# 原始行名
orig_micro_names = df_micro.index.tolist()

# 清洗
cleaned_micro_names = df_micro.index.map(convert_abund_rowname_microbe)

# 保存映射
microbe_name_map = pd.DataFrame({
    'original_name': orig_micro_names,
    'cleaned_name': cleaned_micro_names
})
microbe_name_map.to_csv(cfg.MICROBE_NAME_MAPPING_CSV, index=False)
print("Microbe name mapping saved.")


# 只用 convert_abund_rowname_microbe 处理
df_micro.index = cleaned_micro_names

# 去掉空行名和重复行
df_micro = df_micro[~df_micro.index.isna()]
df_micro = df_micro.loc[~df_micro.index.duplicated(keep='first')]

# 保存新文件
df_micro.to_csv(microbe_abund_new_file)
print(f"Cleaned microbe abundance saved to: {microbe_abund_new_file}")


# ==========================================================
# 5. 清洗通路丰度表
# ==========================================================
df_pathway = pd.read_csv(pathway_abund_file, index_col=0)


df_pathway = df_pathway[~df_pathway.index.astype(str).str.contains(r"UNINTEGRATED|UNMAPPED|\|")]
orig_pathway_names = df_pathway.index.tolist()


# 清洗
cleaned_pathway_names = df_pathway.index.astype(str).map(convert_abund_rowname_pathway)

# 保存映射
pathway_name_map = pd.DataFrame({
    'original_name': orig_pathway_names,
    'cleaned_name': cleaned_pathway_names
})
pathway_name_map.to_csv(cfg.PATHWAY_NAME_MAPPING_CSV, index=False)
print("Pathway name mapping saved.")

#df_pathway = df_pathway[~df_pathway.index.astype(str).str.contains(r"UNINTEGRATED|UNMAPPED|\|")]
# 更新索引
df_pathway.index = cleaned_pathway_names

# 去掉空行名和重复行
df_pathway = df_pathway[~df_pathway.index.isna()]
df_pathway = df_pathway.loc[~df_pathway.index.duplicated(keep='first')]

# 保存新文件
df_pathway.to_csv(pathway_abund_new_file)
print(f"Cleaned pathway abundance saved to: {pathway_abund_new_file}")


Microbe name mapping saved.
Cleaned microbe abundance saved to: D:\coursework\研究生\code\MGPAN\data\NielsenHB_2014\relative_abundance_cleaned1.csv
Pathway name mapping saved.
Cleaned pathway abundance saved to: D:\coursework\研究生\code\MGPAN\data\NielsenHB_2014\pathway_abundance_cleaned1.csv


## 6. Generate pathway-pathway edges


In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os

# -----------------------------
# 数据路径
#V2018+L2016
TEST_ABUND = cfg.PP_INPUT_CSV
OUT_DIR = cfg.PP_OUTPUT_DIR
os.makedirs(OUT_DIR, exist_ok=True)

df_abund = pd.read_csv(TEST_ABUND, index_col=0)
# 去掉包含 UNINTEGRATED / UNMAPPED / | 的行
df_abund = df_abund[~df_abund.index.astype(str).str.contains(r"UNINTEGRATED|UNMAPPED|\|")]

# 去掉冒号及其后面的内容
df_abund.index = df_abund.index.astype(str).str.split(":", n=1).str[0]
samples = df_abund.columns.tolist()


In [9]:
import os
import numpy as np
import pandas as pd

# ==========================================================
# P-P 网络生成函数（单样本，强制正负各半）
# ==========================================================
def generate_pp_network_logratio_topk_signed(
    sample_vector,
    pathway_names,
    gamma=cfg.PP_GAMMA,
    top_k=cfg.PP_TOP_K
):
    """
    单样本 P-P 网络（对称、有符号、稀疏）

    权重 [-1,1]：
        正值 -> 丰度相似
        负值 -> 丰度差异大

    Top-K 保留时，每个节点正负各占一半
    """

    EPS = cfg.EDGE_EPS
    n = len(sample_vector)
    if n < 2:
        return pd.DataFrame(columns=["OTU1_clean", "OTU2_clean", "weight"])

    # Step 1: log 转换
    log_abund = np.log(sample_vector + EPS)

    # Step 2: z-score 标准化
    if np.std(log_abund) < 1e-9:
        z_abund = log_abund - np.mean(log_abund)
    else:
        z_abund = (log_abund - np.mean(log_abund)) / np.std(log_abund)

    # Step 3: 差异矩阵
    diff_matrix = np.abs(z_abund[:, None] - z_abund[None, :])

    # Step 4: tanh 映射 [-1,1]
    max_diff = np.max(diff_matrix)
    if max_diff == 0:
        return pd.DataFrame(columns=["OTU1_clean", "OTU2_clean", "weight"])
    weight_matrix = np.tanh(gamma * (0.5 - diff_matrix / max_diff))

    # Step 5: 上三角边表
    iu = np.triu_indices(n, k=1)
    df_edges = pd.DataFrame({
        "OTU1_clean": [pathway_names[i] for i in iu[0]],
        "OTU2_clean": [pathway_names[j] for j in iu[1]],
        "weight": weight_matrix[iu]
    })

    # Step 6: Top-K 正负各半保留
    edges_filtered = []
    for node in pathway_names:
        mask = (df_edges["OTU1_clean"] == node) | (df_edges["OTU2_clean"] == node)
        sub = df_edges[mask].sort_values(by="weight", key=lambda x: x.abs(), ascending=False)

        # 正负各半
        sub_pos = sub[sub["weight"] > 0].head(top_k // 2)
        sub_neg = sub[sub["weight"] < 0].head(top_k // 2)
        edges_filtered.append(pd.concat([sub_pos, sub_neg]))

    df_edges = pd.concat(edges_filtered).drop_duplicates(subset=["OTU1_clean", "OTU2_clean"])

    # Step 7: 无向唯一化
    df_edges[["n1", "n2"]] = np.sort(df_edges[["OTU1_clean", "OTU2_clean"]].values, axis=1)
    df_edges = df_edges.drop(columns=["OTU1_clean", "OTU2_clean"]).rename(columns={"n1": "OTU1_clean", "n2": "OTU2_clean"})
    df_edges = df_edges.groupby(["OTU1_clean", "OTU2_clean"], as_index=False)["weight"].mean()

    df_edges = df_edges[df_edges["weight"] != 0]
    df_edges.reset_index(drop=True, inplace=True)

    return df_edges


# ==========================================================
# 主循环示例
# ==========================================================
OUT_DIR = cfg.PP_OUTPUT_DIR
os.makedirs(OUT_DIR, exist_ok=True)

gamma = cfg.PP_GAMMA
top_k = cfg.PP_TOP_K

total_edges = 0
total_nodes = 0
num_samples = 0
ABUND_TH = cfg.PP_ABUNDANCE_THRESHOLD
for sample_id in samples:
    abund_vec = df_abund[sample_id].values

    # 只保留非零丰度通路
    nonzero_idx = np.where(abund_vec > ABUND_TH)[0]
    abund_vec_nonzero = abund_vec[nonzero_idx]
    pathway_names_nonzero = df_abund.index[nonzero_idx].tolist()

    # 如果不足 2 个节点 → 输出一个空图，不跳过
    if len(nonzero_idx) < 2:
        df_edges = pd.DataFrame(columns=["OTU1_clean", "OTU2_clean", "weight"])
        out_file = os.path.join(OUT_DIR, f"{sample_id}_PP_edges.csv")
        df_edges.to_csv(out_file, index=False)

        print(f"Sample {sample_id}: only {len(nonzero_idx)} nonzero pathways → saved 0-edge graph.")
        
        total_edges += 0
        total_nodes += len(pathway_names_nonzero)
        num_samples += 1
        continue

    # 正常生成 P-P 图（>=2 节点）
    df_edges = generate_pp_network_logratio_topk_signed(
        sample_vector=abund_vec_nonzero,
        pathway_names=pathway_names_nonzero,
        gamma=gamma,
        top_k=top_k
    )

    # 保存
    out_file = os.path.join(OUT_DIR, f"{sample_id}_PP_edges.csv")
    df_edges.to_csv(out_file, index=False)
    print(f"Sample {sample_id} saved: {len(df_edges)} edges, pathways: {len(pathway_names_nonzero)}")

    total_edges += len(df_edges)
    total_nodes += len(pathway_names_nonzero)
    num_samples += 1

# 平均边数和节点数
if num_samples > 0:
    avg_edges = total_edges / num_samples
    avg_nodes = total_nodes / num_samples
    print(f"\nAverage P-P edges across {num_samples} samples: {avg_edges:.2f}")
    print(f"Average pathways (nodes) across {num_samples} samples: {avg_nodes:.2f}")
else:
    print("No samples processed.")


Sample MH0001 saved: 205 edges, pathways: 64
Sample MH0002 saved: 222 edges, pathways: 74
Sample MH0003 saved: 289 edges, pathways: 90
Sample MH0004 saved: 208 edges, pathways: 64
Sample MH0005 saved: 215 edges, pathways: 69
Sample MH0006 saved: 260 edges, pathways: 83
Sample MH0007 saved: 218 edges, pathways: 70
Sample MH0008 saved: 295 edges, pathways: 93
Sample MH0009 saved: 177 edges, pathways: 56
Sample MH0010 saved: 311 edges, pathways: 99
Sample MH0011 saved: 243 edges, pathways: 76
Sample MH0012 saved: 241 edges, pathways: 74
Sample MH0013 saved: 86 edges, pathways: 29
Sample MH0014 saved: 310 edges, pathways: 98
Sample MH0015 saved: 215 edges, pathways: 67
Sample MH0016 saved: 250 edges, pathways: 83
Sample MH0017 saved: 173 edges, pathways: 57
Sample MH0018 saved: 230 edges, pathways: 72
Sample MH0019 saved: 291 edges, pathways: 95
Sample MH0020 saved: 216 edges, pathways: 76
Sample MH0021 saved: 281 edges, pathways: 89
Sample MH0022 saved: 64 edges, pathways: 23
Sample MH002

## 7. Generate microbe-microbe edges


In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os
import re
#V2018+L2016
# -----------------------------
# 数据路径
TEST_ABUND = cfg.MM_INPUT_CSV
OUT_DIR = cfg.MM_OUTPUT_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# 读取微生物丰度矩阵（行=微生物, 列=样本）
df_abund = pd.read_csv(TEST_ABUND, index_col=0)
microbes = df_abund.index.tolist()
samples = df_abund.columns.tolist()

In [11]:


# ==========================================================
# 一、微生物名清洗函数
# ==========================================================
def convert_abund_rowname_microbe(rowname):
    if isinstance(rowname, str) and rowname.count('g__') == 1 and rowname.count('s__') == 1 and '|' not in rowname:
        return rowname.strip()

    match_g = re.search(r'g__[^|.]+', rowname)
    match_s = re.search(r's__[^|.]+', rowname)
    if match_g and match_s:
        return f"{match_g.group(0)}.{match_s.group(0)}"
    elif match_g:
        return match_g.group(0)
    else:
        return None

# ==========================================================
# 二、M–M 网络生成函数（保留负值 + 去重 + Top-k）
# ==========================================================

def generate_mm_network_logratio_topk_signed(sample_vector, microbe_names, top_k=cfg.MM_TOP_K, gamma=cfg.MM_GAMMA):
    """
    单样本微生物-微生物网络
    输出 [-1,1] 权重：
    - 正值：丰度相似，可能协同
    - 负值：丰度差异大，可能互斥
    Top-K 正负各半保留。
    """

    EPS = cfg.EDGE_EPS
    n = len(sample_vector)
    if n < 2:
        return pd.DataFrame(columns=["OTU1_gs", "OTU2_gs", "weight"])

    # Step 1: log 转换
    log_abund = np.log(sample_vector + EPS)

    # Step 2: z-score 标准化
    if np.std(log_abund) < 1e-9:
        z_abund = log_abund - np.mean(log_abund)
    else:
        z_abund = (log_abund - np.mean(log_abund)) / np.std(log_abund)

    # Step 3: 差异矩阵
    diff_matrix = np.abs(z_abund[:, None] - z_abund[None, :])

    # Step 4: tanh 映射到 [-1,1]，增强对比
    max_diff = np.max(diff_matrix)
    if max_diff == 0:
        return pd.DataFrame(columns=["OTU1_gs", "OTU2_gs", "weight"])
    weight_matrix = np.tanh(gamma * (0.5 - diff_matrix / max_diff))

    # Step 5: 上三角边表
    iu = np.triu_indices(n, k=1)
    df_edges = pd.DataFrame({
        "OTU1_gs": [microbe_names[i] for i in iu[0]],
        "OTU2_gs": [microbe_names[j] for j in iu[1]],
        "weight": weight_matrix[iu]
    })

    # Step 6: Top-K 正负各半
    edges_filtered = []
    for node in microbe_names:
        mask = (df_edges["OTU1_gs"] == node) | (df_edges["OTU2_gs"] == node)
        sub = df_edges[mask].sort_values(by="weight", key=lambda x: x.abs(), ascending=False)

        sub_pos = sub[sub["weight"] > 0].head(top_k // 2)
        sub_neg = sub[sub["weight"] < 0].head(top_k // 2)
        edges_filtered.append(pd.concat([sub_pos, sub_neg]))

    # edges_filtered = []
    # for node in microbe_names:
    #     mask = (df_edges["OTU1_gs_gs"] == node) | (df_edges["OTU2_gs"] == node)
    #     sub = df_edges[mask].sort_values(by="weight", key=lambda x: x.abs(), ascending=False)
    #     edges_filtered.append(sub.head(top_k))

    df_topk = pd.concat(edges_filtered).drop_duplicates(subset=["OTU1_gs", "OTU2_gs"])

    # Step 7: 无向唯一化
    df_topk[["n1", "n2"]] = np.sort(df_topk[["OTU1_gs", "OTU2_gs"]].values, axis=1)
    df_topk = df_topk.drop(columns=["OTU1_gs", "OTU2_gs"]).rename(columns={"n1": "OTU1_gs", "n2": "OTU2_gs"})
    df_topk = df_topk.groupby(["OTU1_gs", "OTU2_gs"], as_index=False)["weight"].mean()

    # Step 8: 输出
    df_topk = df_topk[df_topk["weight"] != 0]
    df_topk.reset_index(drop=True, inplace=True)
    return df_topk

# ==========================================================
# 三、主循环示例
# ==========================================================
OUT_DIR = cfg.MM_OUTPUT_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# 假设 df_abund 已经存在
# df_abund = pd.read_csv("microbe_abundance.csv", index_col=0)
df_abund.index = df_abund.index.map(convert_abund_rowname_microbe)
df_abund = df_abund[~df_abund.index.isna()]
df_abund = df_abund.loc[~df_abund.index.duplicated(keep='first')]

top_k = cfg.MM_TOP_K
gamma = cfg.MM_GAMMA
total_edges, total_nodes, num_samples = 0, 0, 0

ABUND_TH = cfg.MM_ABUNDANCE_THRESHOLD
for sample_id in df_abund.columns:
    abund_vec = df_abund[sample_id].values
    nonzero_idx = np.where(abund_vec > ABUND_TH)[0]
    abund_vec_nonzero = abund_vec[nonzero_idx]
    microbe_names_nonzero = df_abund.index[nonzero_idx].tolist()

    # 不跳过样本
    if len(nonzero_idx) < 2:
        df_edges = pd.DataFrame(columns=["OTU1_gs", "OTU2_gs", "weight"])
        out_file = os.path.join(OUT_DIR, f"{sample_id}_MM_edges.csv")
        df_edges.to_csv(out_file, index=False)
        print(f"Sample {sample_id}: only {len(nonzero_idx)} nonzero microbes → saved 0-edge graph.")
        
        total_edges += 0
        total_nodes += len(microbe_names_nonzero)
        num_samples += 1
        continue

    df_edges = generate_mm_network_logratio_topk_signed(
        sample_vector=abund_vec_nonzero,
        microbe_names=microbe_names_nonzero,
        top_k=top_k,
        gamma=gamma
    )

    out_file = os.path.join(OUT_DIR, f"{sample_id}_MM_edges.csv")
    df_edges.to_csv(out_file, index=False)
    print(f"Sample {sample_id}: {len(df_edges)} edges, microbes: {len(microbe_names_nonzero)}")

    total_edges += len(df_edges)
    total_nodes += len(microbe_names_nonzero)
    num_samples += 1
print(num_samples)

if num_samples > 0:
    print(f"\nAverage edges per sample: {total_edges / num_samples:.2f}")
    print(f"Average microbes per sample: {total_nodes / num_samples:.2f}")
else:
    print("No samples processed.")

Sample MH0001: 122 edges, microbes: 74
Sample MH0002: 133 edges, microbes: 80
Sample MH0003: 147 edges, microbes: 87
Sample MH0004: 150 edges, microbes: 91
Sample MH0005: 142 edges, microbes: 84
Sample MH0006: 119 edges, microbes: 72
Sample MH0007: 180 edges, microbes: 109
Sample MH0008: 126 edges, microbes: 76
Sample MH0009: 146 edges, microbes: 89
Sample MH0010: 98 edges, microbes: 58
Sample MH0011: 134 edges, microbes: 80
Sample MH0012: 139 edges, microbes: 86
Sample MH0013: 181 edges, microbes: 108
Sample MH0014: 120 edges, microbes: 72
Sample MH0015: 145 edges, microbes: 89
Sample MH0016: 100 edges, microbes: 62
Sample MH0017: 158 edges, microbes: 97
Sample MH0018: 108 edges, microbes: 65
Sample MH0019: 111 edges, microbes: 68
Sample MH0020: 86 edges, microbes: 51
Sample MH0021: 134 edges, microbes: 81
Sample MH0022: 138 edges, microbes: 83
Sample MH0023: 141 edges, microbes: 85
Sample MH0024: 72 edges, microbes: 44
Sample MH0025: 144 edges, microbes: 88
Sample MH0026: 75 edges, m